# Scratch – repr & visualisation testing

Temporary notebook for testing `_repr_html_` / `__repr__` across all classes and visualisation helpers like `coverage_bar()` and `tree()`.

**Delete when done.**

In [2]:
from datetime import datetime, timezone

import numpy as np

from timedatamodel import (
    AggregationMethod,
    CoverageBar,
    DataType,
    Dimension,
    Frequency,
    GeoLocation,
    HierarchicalTimeSeries,
    HierarchyNode,
    TimeSeries,
    TimeSeriesArray,
    TimeSeriesCollection,
    TimeSeriesTable,
    TimeSeriesType,
)

## Helper – generate hourly timestamps

In [3]:
def hourly_ts(n=24, start="2024-01-15"):
    base = datetime.fromisoformat(f"{start}T00:00:00+00:00")
    from datetime import timedelta
    return [base + timedelta(hours=i) for i in range(n)]

---
## 1. TimeSeries

In [4]:
ts = TimeSeries(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=hourly_ts(24),
    values=[120 + 5 * np.sin(i) for i in range(24)],
    name="power",
    unit="MW",
    data_type=DataType.MEASUREMENT,
    description="Hourly power output from wind farm Alpha",
)
ts

Name,power
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,MW
Data type,MEASUREMENT
Description,Hourly power output from wind farm Alpha
timestamp,power
2024-01-15,120.0
2024-01-15 01:00,124.207
2024-01-15 02:00,124.546


In [ ]:
print(ts)

### TimeSeries – empty

In [5]:
ts_empty = TimeSeries(Frequency.PT1H, timestamps=[], values=[], name="empty")
ts_empty

TimeSeries
┌───────────────────────────┐
│  Name:             empty  │
│  Length:           0      │
│  Frequency:        PT1H   │
│  Timezone:         UTC    │
├───────────────────────────┤
│  (empty)                  │
└───────────────────────────┘

### TimeSeries – with NaN gaps

In [6]:
vals_gap = [float(i) for i in range(24)]
for i in [5, 6, 7, 15, 16]:
    vals_gap[i] = None

ts_gap = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=vals_gap,
    name="gappy_signal",
    unit="kW",
)
ts_gap

Name,gappy_signal
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,kW
timestamp,gappy_signal
2024-01-15,0.0
2024-01-15 01:00,1.0
2024-01-15 02:00,2.0
…,…
2024-01-15 21:00,21.0


### TimeSeries – short (no truncation)

In [7]:
ts_short = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(5),
    values=[1.0, 2.0, 3.0, 4.0, 5.0],
    name="short",
)
ts_short

Name,short
Length,5
Frequency,PT1H
Timezone,UTC (+00:00)
timestamp,short
2024-01-15,1.0
2024-01-15 01:00,2.0
2024-01-15 02:00,3.0


### TimeSeries – with location and labels

In [8]:
ts_loc = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=[100 + i * 0.5 for i in range(24)],
    name="temperature",
    unit="°C",
    location=GeoLocation(latitude=59.91, longitude=10.75),
    labels={"source": "met.no", "station": "Oslo-Blindern"},
    data_type=DataType.MEASUREMENT,
    description="Air temperature at Blindern station",
)
ts_loc

Name,temperature
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,°C
Data type,MEASUREMENT
Location,"59.91°N, 10.75°E"
Description,Air temperature at Blindern station
Labels,"{'source': 'met.no', 'station': 'Oslo-Blindern'}"
timestamp,temperature
2024-01-15,100.0


---
## 2. TimeSeriesTable

In [10]:
timestamps = hourly_ts(24)
tbl = TimeSeriesTable(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=timestamps,
    values=np.column_stack([
        [120 + 5 * np.sin(i) for i in range(24)],
        [80 + 3 * np.cos(i) for i in range(24)],
        [200 + 10 * np.sin(i + 1) for i in range(24)],
    ]),
    names=["wind_farm_A", "wind_farm_B", "solar_park_C"],
    units=["MW", "MW", "MW"],
    data_types=[DataType.MEASUREMENT, DataType.MEASUREMENT, DataType.FORECAST],
)
tbl

TimeSeriesTable
┌────────────────────────────────────────────────────────────┐
│  Names:            wind_farm_A, wind_farm_B, solar_park_C  │
│  Length:           24 × 3                                  │
│  Frequency:        PT1H                                    │
│  Timezone:         UTC (+00:00)                            │
│  Unit:             MW, MW, MW                              │
│  Data type:        MEASUREMENT, MEASUREMENT, FORECAST      │
├────────────────────────────────────────────────────────────┤
│                    wind_farm_A  wind_farm_B  solar_park_C  │
│  2024-01-15              120.0         83.0       208.415  │
│  2024-01-15 01:00      124.207      81.6209       209.093  │
│  2024-01-15 02:00      124.546      78.7516       201.411  │
│  ...                       ...          ...           ...  │
│  2024-01-15 21:00      124.183      78.3568       199.911  │
│  2024-01-15 22:00      119.956      77.0001       191.538  │
│  2024-01-15 23:00      115.769      78.4015       190.944  │
└────────────────────────────────────────────────────────────┘

In [ ]:
print(tbl)

---
## 3. TimeSeriesCollection

In [11]:
ts_a = TimeSeries(
    Frequency.PT1H, timestamps=hourly_ts(24), values=[float(i) for i in range(24)],
    name="series_A", unit="MW",
)
ts_b = TimeSeries(
    Frequency.P1D,
    timestamps=[datetime.fromisoformat(f"2024-01-{d:02d}T00:00:00+00:00") for d in range(1, 8)],
    values=[100.0, 105.0, 98.0, None, 110.0, 115.0, 108.0],
    name="series_B", unit="GWh",
)

coll = TimeSeriesCollection(
    [ts_a, ts_b, tbl],
    name="My Collection",
    description="A mix of different series",
)
coll

name,type,freq,tz,length,begin,end
series_A,TimeSeries,PT1H,UTC,24,2024-01-15,2024-01-15 23:00
series_B,TimeSeries,P1D,UTC,7,2024-01-01,2024-01-07
"wind_farm_A,wind_farm_B,solar_park_C",TimeSeriesTable,PT1H,UTC,24,2024-01-15,2024-01-15 23:00


In [12]:
print(coll)

┌────────────────────────────────── TimeSeriesCollection: My Collection ───────────────────────────────────┐
│  name                                  type             freq  tz   length  begin       end               │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  series_A                              TimeSeries       PT1H  UTC  24      2024-01-15  2024-01-15 23:00  │
│  series_B                              TimeSeries       P1D   UTC  7       2024-01-01  2024-01-07        │
│  wind_farm_A,wind_farm_B,solar_park_C  TimeSeriesTable  PT1H  UTC  24      2024-01-15  2024-01-15 23:00  │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘


---
## 4. TimeSeriesArray (N-dimensional)

In [13]:
time_labels = hourly_ts(12)
scenarios = ["low", "mid", "high"]

arr = TimeSeriesArray(
    Frequency.PT1H,
    name="price_scenarios",
    unit="EUR/MWh",
    data_type=DataType.SCENARIO,
    dimensions=[
        Dimension(name="timestamp", labels=time_labels),
        Dimension(name="scenario", labels=scenarios),
    ],
    values=np.random.default_rng(42).uniform(30, 80, size=(12, 3)),
)
arr

TimeSeriesArray
┌────────────────────────────────────────────────┐
│  Dimensions:       timestamp: 12, scenario: 3  │
│  Shape:            (12, 3)                     │
│  Frequency:        PT1H                        │
│  Timezone:         UTC (+00:00)                │
│  Name:             price_scenarios             │
│  Unit:             EUR/MWh                     │
│  Data type:        SCENARIO                    │
└────────────────────────────────────────────────┘

In [ ]:
print(arr)

### TimeSeriesArray – 1D

In [14]:
arr_1d = TimeSeriesArray(
    Frequency.PT1H,
    name="load",
    unit="MW",
    dimensions=[Dimension(name="timestamp", labels=hourly_ts(8))],
    values=np.array([100, 105, 110, 108, 103, 99, 95, 92], dtype=float),
)
arr_1d

Dimensions,timestamp: 8
Shape,"(8,)"
Frequency,PT1H
Timezone,UTC (+00:00)
Name,load
Unit,MW
timestamp,load
2024-01-15 00:00:00+00:00,100.0
2024-01-15 01:00:00+00:00,105.0
2024-01-15 02:00:00+00:00,110.0
…,…


---
## 5. CoverageBar

In [15]:
ts_gap.coverage_bar()

gappy_signal  █████░░░███████░░███████
              2024-01-15  2024-01-15 23:00

In [16]:
print(ts_gap.coverage_bar())

gappy_signal  █████░░░███████░░███████
              2024-01-15  2024-01-15 23:00


In [17]:
tbl.coverage_bar()

wind_farm_A   ████████████████████████
wind_farm_B   ████████████████████████
solar_park_C  ████████████████████████
              2024-01-15  2024-01-15 23:00

In [18]:
print(tbl.coverage_bar())

wind_farm_A   ████████████████████████
wind_farm_B   ████████████████████████
solar_park_C  ████████████████████████
              2024-01-15  2024-01-15 23:00


---
## 6. HierarchicalTimeSeries & tree()

In [19]:
timestamps = hourly_ts(24)
rng = np.random.default_rng(0)

def make_ts(name):
    return TimeSeries(
        Frequency.PT1H, timestamps=timestamps,
        values=rng.uniform(50, 200, 24).tolist(),
        name=name, unit="MW", data_type=DataType.MEASUREMENT,
    )

tree_dict = {
    "total": {
        "Norway": {"Bergen": "bergen", "Oslo": "oslo", "Trondheim": "trondheim"},
        "Sweden": {"Stockholm": "stockholm", "Gothenburg": "gothenburg"},
    }
}
series_map = {
    "bergen": make_ts("Bergen"),
    "oslo": make_ts("Oslo"),
    "trondheim": make_ts("Trondheim"),
    "stockholm": make_ts("Stockholm"),
    "gothenburg": make_ts("Gothenburg"),
}

hts = HierarchicalTimeSeries.from_dict(
    tree_dict, series_map,
    levels=["region", "country", "city"],
    name="Nordic Wind Power",
    description="Hierarchical wind power across Nordic cities",
    aggregation=AggregationMethod.SUM,
)
hts

HierarchicalTimeSeries
┌─────────────────────────────────────────────────────────────┐
│  Name:             Nordic Wind Power                        │
│  Levels:           region, country, city                    │
│  Nodes:            16 (5 leaves)                            │
│  Frequency:        PT1H                                     │
│  Timezone:         UTC (+00:00)                             │
│  Unit:             MW                                       │
│  Aggregation:      sum                                      │
├─────────────────────────────────────────────────────────────┤
│  name        level    length  begin       end               │
├─────────────────────────────────────────────────────────────┤
│  Bergen      country  24      2024-01-15  2024-01-15 23:00  │
│  Oslo        country  24      2024-01-15  2024-01-15 23:00  │
│  Trondheim   country  24      2024-01-15  2024-01-15 23:00  │
│  Stockholm   country  24      2024-01-15  2024-01-15 23:00  │
│  Gothenburg  country  24      2024-01-15  2024-01-15 23:00  │
└─────────────────────────────────────────────────────────────┘

In [ ]:
print(hts)

### HierarchyTree – HTML

In [20]:
hts.tree()

└── total (region)
    └── total (region)
        ├── Norway (region)
        │   └── Norway (region)
        │       ├── Bergen (region)
        │       │   └── Bergen [24 pts]
        │       ├── Oslo (region)
        │       │   └── Oslo [24 pts]
        │       └── Trondheim (region)
        │           └── Trondheim [24 pts]
        └── Sweden (region)
            └── Sweden (region)
                ├── Stockholm (region)
                │   └── Stockholm [24 pts]
                └── Gothenburg (region)
                    └── Gothenburg [24 pts]

In [ ]:
print(hts.tree())

In [22]:
ts.df

,power
timestamp,
2024-01-15 00:00:00+00:00,120.000000
2024-01-15 01:00:00+00:00,124.207355
2024-01-15 02:00:00+00:00,124.546487
2024-01-15 03:00:00+00:00,120.705600
2024-01-15 04:00:00+00:00,116.215988
2024-01-15 05:00:00+00:00,115.205379
2024-01-15 06:00:00+00:00,118.602923
2024-01-15 07:00:00+00:00,123.284933
2024-01-15 08:00:00+00:00,124.946791


---
## 7. Wide TimeSeriesTable (many columns)

Test whether the repr truncates columns when there are too many to fit.

In [23]:
n_cols = 20
timestamps = hourly_ts(48)
rng = np.random.default_rng(7)

tbl_wide = TimeSeriesTable(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=timestamps,
    values=rng.uniform(50, 300, size=(48, n_cols)),
    names=[f"turbine_{i:02d}" for i in range(n_cols)],
    units=["MW"] * n_cols,
)
tbl_wide

TimeSeriesTable
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  Names:            turbine_00, turbine_01, turbine_02, turbine_03, turbine_04, turbine_05, turbine_06, turbine_07, turbine_08, turbine_09, turbine_10, turbine_11, turbine_12, turbine_13, turbine_14, turbine_15, turbine_16, turbine_17, turbine_18, turbine_19  │
│  Length:           48 × 20                                                                                                                                                                                                                                         │
│  Frequency:        PT1H                                                                                                                                                                                                                                            │
│  Timezone:         UTC (+00:00)                                                                                                                                                                                                                                    │
│  Unit:             MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW                                                                                                                                                                  │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                    turbine_00  turbine_01  turbine_02  turbine_03  turbine_04  turbine_05  turbine_06  turbine_07  turbine_08  turbine_09  turbine_10  turbine_11  turbine_12  turbine_13  turbine_14  turbine_15  turbine_16  turbine_17  turbine_18  turbine_19  │
│  2024-01-15           206.274     274.303     243.921     106.302     125.042     268.388     51.3163     255.307     249.267     166.984     125.758     119.606     113.717     161.269     176.137     188.374     298.875     248.165     205.545      297.24  │
│  2024-01-15 01:00     103.827      90.053     203.135     60.9855     58.9201     178.722     166.552     279.292     207.307     178.529     174.218     111.879     52.9485     98.1005     223.008     100.152     142.384     50.9336     257.512     88.6153  │
│  2024-01-15 02:00       116.9     270.083     177.448     261.788     209.929     235.443     72.8739     185.286     176.943     267.835     140.316     199.546     64.8129     146.908     130.759     87.5499     254.085     144.862     294.687     197.498  │
│  ...                      ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...  │
│  2024-01-16 21:00     115.077      195.91     288.058     254.283     210.182     179.705      53.783     272.679     66.7135     82.2703     223.268     253.036     246.638     163.101     94.2639     58.8256     262.845     294.424     111.034     266.519  │
│  2024-01-16 22:00     150.633     127.836     176.289      52.042     160.679     99.8887     228.114     175.643     267.213     274.162      129.19     120.096     284.783     175.999     74.1289     61.4713     297.606     194.791     63.6232     177.814  │
│  2024-01-16 23:00     176.026     158.847      287.82     287.193     81.2419     87.9302     151.872     258.507     205.938     232.458     194.015     102.514     254.441     53.5052     224.621     98.5849     167.773     293.783     184.759     64.2372  │
└──────────────────────────────────────

In [24]:
print(tbl_wide)

TimeSeriesTable
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  Names:            turbine_00, turbine_01, turbine_02, turbine_03, turbine_04, turbine_05, turbine_06, turbine_07, turbine_08, turbine_09, turbine_10, turbine_11, turbine_12, turbine_13, turbine_14, turbine_15, turbine_16, turbine_17, turbine_18, turbine_19  │
│  Length:           48 × 20                                                                                                                                                                                                                                         │
│  Frequency:        PT1H                                                                                                                                                                          

### Same data as pandas DataFrame (for comparison)

In [ ]:
tbl_wide.df

---
## 8. Side-by-side: light vs dark

Quick way to preview both colour schemes without changing your OS setting — wrap the HTML repr in a `<div>` that forces a colour scheme.

In [21]:
from IPython.display import HTML

light = f'<div style="color-scheme: light; padding: 10px;">{ts._repr_html_()}</div>'
dark = f'<div style="color-scheme: dark; background: #0f172a; padding: 10px; border-radius: 8px;">{ts._repr_html_()}</div>'

HTML(f"<h4>Light mode</h4>{light}<br><h4>Dark mode</h4>{dark}")

Name,power
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,MW
Data type,MEASUREMENT
Description,Hourly power output from wind farm Alpha
timestamp,power
2024-01-15,120.0
2024-01-15 01:00,124.207
2024-01-15 02:00,124.546
